# Day 52：微调数据集构建与 Self-Instruct 合成流水线（Jupyter Notebook 实战手册）

> **所属阶段**：第 5 阶段 · 第五阶段：模型微调与部署 ｜ **第 8 周**：模型微调、本地推理部署与安全合规  
> **GitLab 工单**：`STARMIND-052` ｜ **开发分支**：`feat/day52-dataset-pipeline-synth`  
> **配套 HTML5 课件**：`courseware/day52/index.html` ｜ **独立版本快照**：`versions/day52/`

---

## 🎙️ 老陈开场旁白

各位早，今天讲整个微调链路里最值钱、也最容易被忽视的一环——数据集工程。行业里有句大实话：'Garbage in, garbage out'，你喂进去一堆带乱码、带'作为一个AI语言模型'废话、互相矛盾的脏数据，给再牛的 A100 训出来也是个智障。今天咱不靠纯手工搬砖，先写一套工业级清洗去重管道（支持 Alpaca 和 ShareGPT 双格式互转），再写一个 Self-Instruct 垂直领域合成器，把星辰数科的工单、发票、部署、SLA 四大客服场景直接合成 500 条高纯度训练语料。

### 承上启下
- **昨日基线**：昨天我们算清楚了账：星辰政企客服场景采用'RAG + Qwen2.5-7B LoRA'混合架构，单张 4090 花 4 块钱就能跑一轮。但决策引擎也给了预警——目前人工整理的高质量样本不到 100 条，直接训必过拟合。
- **明日预告**：明天（Day 53）我们把今天生成的 500 条高纯度数据集喂进 LLaMA-Factory，自动生成标准微调 YAML 配置，并深入学习 lr/epoch/batch_size 调优与 Loss 曲线诊断。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day52'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-052）

星辰数科准备微调专属客服模型，历史工单导出的原始数据存在大量重复问答、空白回复、包含客户手机号隐私以及非标准格式问题，无法直接用于 LLaMA-Factory 训练。

**今日迭代目标：**
- 实现支持 Alpaca 与 ShareGPT 双向转换及 Schema 校验的标准化工具。
- 实现多阶段清洗管道：隐私脱敏、长度/黑名单词过滤、精确指纹去重与 Jaccard 近重复过滤。
- 构建可离线确定性运行的星辰客服 500 条垂直领域语料合成脚本，并按 9:1 切分 train/eval 集。

**验收标准：**
- 清洗管道能自动剔除长度过短、含'作为一个AI'腔调、完全重复及高相似度近重复样本，并输出清洗漏斗统计。
- 合成器稳定输出 >= 500 条不重复、结构合法的 Alpaca/ShareGPT 语料。
- 脱敏过滤器确保合成与清洗后的数据集不包含真实 11 位手机号。

---

## 📚 课堂核心技术笔记

### 1. 1. Alpaca vs ShareGPT 怎么选？

如果你的任务是单轮指令（如工单分类、SQL 生成、固定文本抽取），用 Alpaca 格式最直观，包含 `instruction`、`input`、`output` 和可选 `system` 字段。如果是智能客服、多轮问诊、带工具调用（Function Calling）的 Agent 微调，必须用 ShareGPT 格式，用 `conversations` 列表串联多轮 `human` / `gpt`（或 `function_call` / `observation`）。

> **⚠️ 老陈避坑**：坑点：ShareGPT 格式中如果连续出现两条 `human` 或第一条是 `gpt`，LLaMA-Factory 加载时会直接报错退出，必须做角色交替校验。

### 2. 2. 为什么要做近似去重（Near-Deduplication）？

真实业务日志中经常出现大量'只改了个订单号或用户名，其余一字不差'的重复工单。如果直接进训练集，模型会迅速背诵这些高频模板导致验证集 Loss 飙升（过拟合）。使用字符级 N-gram 的 Jaccard 相似度（或大规模数据用 MinHash LSH）剔除相似度 > 0.85 的样本，能大幅提升模型泛化能力。

> **⚠️ 老陈避坑**：坑点：只做精确字符串去重，忽略了标点空格差异和近重复样本，结果 5000 条数据里其实只有 300 条有效信息。

### 3. 3. SFT 数据量的黄金法则：少而精胜过多而杂

对于 7B/14B 模型的垂直领域风格微调，500~3000 条经过人工把关的高质量样本，往往比 5 万条爬虫拼凑的数据效果好得多。此外，建议在领域数据中按 5%~10% 比例混入通用指令数据（Replay），防止模型发生'灾难性遗忘'（微调完只会回客服话术，连基本算术和写邮件都不会了）。

> **⚠️ 老陈避坑**：坑点：训练集和验证集没做去重隔离，同一类问题既出现在 train 又出现在 eval，导致评估分数虚高 99%，一上线就翻车。


## 💻 实操步骤 1：步骤一：实现微调语料清洗、脱敏、去重与格式转换管道

- **目标文件**：`versions/day52/src/day52_dataset_cleaner.py`
- **老陈代码旁白**：先搭好咱的'数据质检车间'。不管数据是工单导出的还是大模型合成的，全部过一遍脱敏、黑名单、精确去重和 3-gram 近似去重，再支持一键转成 ShareGPT 格式。


In [ ]:
"""
Day 52 - 模块一：微调数据集清洗、隐私脱敏、近似去重与格式转换工具
"""
from __future__ import annotations

import hashlib
import re
from dataclasses import dataclass, field
from typing import Dict, List, Set, Tuple

PHONE_RE = re.compile(r"(?<!\d)1[3-9]\d{9}(?!\d)")
EMAIL_RE = re.compile(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+")

BANNED_PHRASES = (
    "作为一个AI",
    "作为一名人工智能",
    "我只是一个语言模型",
    "无法为您提供帮助",
)


@dataclass
class CleaningStats:
    """清洗漏斗统计指标"""
    total_input: int = 0
    dropped_invalid: int = 0
    dropped_low_quality: int = 0
    dropped_exact_dup: int = 0
    dropped_near_dup: int = 0
    pii_masked_count: int = 0
    kept_count: int = 0


def mask_pii(text: str) -> Tuple[str, int]:
    """对手机号与邮箱进行隐私脱敏，返回(脱敏后文本, 替换次数)"""
    masked, c1 = PHONE_RE.subn("<PHONE_MASKED>", text)
    masked, c2 = EMAIL_RE.subn("<EMAIL_MASKED>", masked)
    return masked, c1 + c2


def _char_ngrams(text: str, n: int = 3) -> Set[str]:
    cleaned = re.sub(r"\s+", "", text.lower())
    if len(cleaned) < n:
        return {cleaned} if cleaned else set()
    return {cleaned[i : i + n] for i in range(len(cleaned) - n + 1)}


def jaccard_similarity(a: str, b: str, n: int = 3) -> float:
    """计算两段文本的字符 N-gram Jaccard 相似度"""
    sa, sb = _char_ngrams(a, n), _char_ngrams(b, n)
    if not sa and not sb:
        return 1.0
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)


def clean_alpaca_dataset(
    records: List[Dict[str, str]],
    min_output_len: int = 12,
    near_dup_threshold: float = 0.88,
) -> Tuple[List[Dict[str, str]], CleaningStats]:
    """执行五步清洗漏斗：校验 -> 脱敏 -> 质量过滤 -> 精确去重 -> 近似去重"""
    stats = CleaningStats(total_input=len(records))
    seen_hashes: Set[str] = set()
    kept: List[Dict[str, str]] = []
    kept_prompts: List[str] = []

    for row in records:
        inst = str(row.get("instruction", "")).strip()
        inp = str(row.get("input", "")).strip()
        out = str(row.get("output", "")).strip()
        sys_prompt = str(row.get("system", "")).strip()

        # 1. 字段完整性校验
        if not inst or not out:
            stats.dropped_invalid += 1
            continue

        # 2. 隐私脱敏
        inst, m1 = mask_pii(inst)
        inp, m2 = mask_pii(inp)
        out, m3 = mask_pii(out)
        stats.pii_masked_count += m1 + m2 + m3

        # 3. 质量与黑名单过滤
        if len(out) < min_output_len or any(p in out for p in BANNED_PHRASES):
            stats.dropped_low_quality += 1
            continue

        # 4. 精确指纹去重
        norm_key = re.sub(r"\s+", "", f"{inst}|{inp}|{out}").lower()
        fp = hashlib.sha256(norm_key.encode("utf-8")).hexdigest()
        if fp in seen_hashes:
            stats.dropped_exact_dup += 1
            continue

        # 5. 近似去重（仅对 prompt 部分快速比对）
        prompt_text = f"{inst} {inp}".strip()
        is_near_dup = any(
            jaccard_similarity(prompt_text, prev) >= near_dup_threshold
            for prev in kept_prompts[-200:]  # 滑动窗口兼顾精度与速度
        )
        if is_near_dup:
            stats.dropped_near_dup += 1
            continue

        seen_hashes.add(fp)
        kept_prompts.append(prompt_text)
        item = {"instruction": inst, "input": inp, "output": out}
        if sys_prompt:
            item["system"] = sys_prompt
        kept.append(item)

    stats.kept_count = len(kept)
    return kept, stats


def alpaca_to_sharegpt(records: List[Dict[str, str]]) -> List[Dict[str, object]]:
    """将 Alpaca 格式无损转换为 LLaMA-Factory 兼容的 ShareGPT 格式"""
    result: List[Dict[str, object]] = []
    for r in records:
        user_msg = r["instruction"] if not r.get("input") else f"{r['instruction']}\n{r['input']}"
        entry: Dict[str, object] = {
            "conversations": [
                {"from": "human", "value": user_msg},
                {"from": "gpt", "value": r["output"]},
            ]
        }
        if r.get("system"):
            entry["system"] = r["system"]
        result.append(entry)
    return result


if __name__ == "__main__":
    demo = [
        {"instruction": "如何开具增值税专票？", "input": "我手机号13812345678", "output": "您好，请登录星辰数科企业控制台-费用中心-发票管理，填写税号后申请，3个工作日内电子交付。"},
        {"instruction": "如何开具增值税专票？", "input": "我手机号13812345678", "output": "您好，请登录星辰数科企业控制台-费用中心-发票管理，填写税号后申请，3个工作日内电子交付。"},
        {"instruction": "你好", "input": "", "output": "作为一个AI语言模型，我无法回答。"},
    ]
    cleaned, st = clean_alpaca_dataset(demo)
    print("清洗统计:", st)
    print("ShareGPT 转换示例:", alpaca_to_sharegpt(cleaned)[0])


### 🔍 步骤 1 代码深度解析

1. `mask_pii` 使用负向环视正则精准匹配中国大陆 11 位手机号，避免误伤订单号。
2. 近似去重采用 3-gram Jaccard 相似度，并配合滑动窗口保证本地处理几千条语料依然是毫秒级。
3. `alpaca_to_sharegpt` 严格遵循 LLaMA-Factory 的 `from: human/gpt` 规范。

**预期运行输出参考：**
```text
清洗统计: CleaningStats(total_input=3, dropped_invalid=0, dropped_low_quality=1, dropped_exact_dup=1, dropped_near_dup=0, pii_masked_count=2, kept_count=1)
ShareGPT 转换示例: {'conversations': [{'from': 'human', 'value': '如何开具增值税专票？\n我手机号<PHONE_MASKED>'}, {'from': 'gpt', 'value': '您好，请登录星辰数科企业控制台-费用中心-发票管理，填写税号后申请，3个工作日内电子交付。'}]}
```


## 💻 实操步骤 2：步骤二：Self-Instruct 合成 500 条星辰垂直领域客服微调数据集

- **目标文件**：`versions/day52/src/day52_self_instruct_gen.py`
- **老陈代码旁白**：质检车间建好了，现在用 Self-Instruct 思路构造 500 条覆盖星辰数科 ERP、智脑 RAG、私有化部署、账单发票、安全审计五大模块的高质量客服语料，过一遍清洗漏斗后自动切分成 train 和 eval。


In [ ]:
"""
Day 52 - 模块二：Self-Instruct 垂直领域客服数据集生成器（产出 500 条高纯度语料）
"""
from __future__ import annotations

import json
import random
from pathlib import Path
from typing import Dict, List, Tuple

from src.day52_dataset_cleaner import alpaca_to_sharegpt, clean_alpaca_dataset

SYSTEM_PROMPT = (
    "你是杭州星辰数科官方资深技术客服「星辰小助手」。"
    "请以专业、简洁、结构化的中文回复客户，明确给出排查步骤、SLA承诺与工单入口，严禁编造政策。"
)

MODULES = [
    ("星辰智脑RAG知识库", "知识库上传PDF解析失败", "检查PDF是否为扫描件并开启OCR开关，单文件上限50MB，解析异常请附带文档ID提交工单"),
    ("星辰ERP财务套件", "月末结账凭证校验报错", "进入财务总账-凭证检查定位借贷不平条目，核对辅助核算项后重新触发过账"),
    ("星辰私有化推理网关", "vLLM容器启动提示显存不足", "核对gpu-memory-utilization参数建议设为0.88，并确认宿主机无残留僵尸进程占用显存"),
    ("企业账单与发票中心", "申请数电增值税专用发票", "在费用中心-发票抬头维护统一社会信用代码与开户行信息，提交后1~3个工作日发送至财务邮箱"),
    ("星辰MCP工具市场", "钉钉审批插件鉴权401超时", "前往开放平台刷新AppSecret并同步更新星辰凭据管理，确认服务器出口IP已加入白名单"),
    ("星辰权限与SSO单点登录", "新员工飞书扫码无法登录", "请管理员在组织架构同步中触发增量拉取，并为该成员分配基础角色组后重试"),
    ("大模型Token配额中心", "接口返回429触发限流熔断", "当前部门日配额已达85%预警或100%硬熔断，请联系管理员在配额中心临时提额或开启降级路由"),
    ("星辰数据安全审计", "导出脱敏日志与网信办合规证明", "进入安全中心-合规报表选择时间范围即可导出脱敏审计包，标准版支持一键下载备案编号说明"),
    ("星辰工作流编排引擎", "定时任务节点未按Cron触发", "检查时区是否设置为Asia/Shanghai，并查看节点运行历史中的上游超时告警"),
    ("多Agent协作沙箱", "代码执行工具报沙箱权限受限", "星辰沙箱默认禁用外网与敏感系统目录读写，请改用内置白名单依赖包并关闭高危系统调用"),
]

CUSTOMER_ROLES = ["运维工程师", "财务主管", "项目经理", "研发负责人", "信息安全专员"]
URGENCY_TONES = [
    ("常规咨询", "请问怎么规范处理？", "标准工单 2 小时内响应"),
    ("线上紧急", "客户现场正在验收，非常着急，请给最快恢复方案！", "P1 紧急通道 15 分钟内工程师介入"),
    ("批量迁移", "我们正在做全公司 200 人切换，需要注意什么？", "专属实施顾问提供 1 对 1 迁移护航"),
    ("首次接入", "我们是新签约政企客户，第一次配置不太熟悉。", "附赠新客上手操作手册与远程协助预约"),
    ("复盘优化", "上周出现过一次类似问题，如何从根源预防？", "建议开启自动化巡检告警规则避免复发"),
]
ENV_CONTEXTS = [
    "部署环境：阿里云ECS专有云",
    "部署环境：本地机房双卡4090内网",
    "部署环境：星辰SaaS企业旗舰版",
    "部署环境：混合云K8s集群v1.28",
]


def synthesize_customer_service_dataset(
    target_count: int = 500,
    seed: int = 2026,
) -> Tuple[List[Dict[str, str]], List[Dict[str, str]]]:
    """合成指定数量的高质量星辰客服语料，经过清洗漏斗后按 9:1 切分训练/验证集"""
    rng = random.Random(seed)
    raw_records: List[Dict[str, str]] = []

    idx = 0
    for mod_name, issue, action in MODULES:
        for role in CUSTOMER_ROLES:
            for tone_tag, tone_q, sla in URGENCY_TONES:
                for env in ENV_CONTEXTS:
                    idx += 1
                    ticket_code = f"CS-{idx:04d}"
                    instruction = (
                        f"【{mod_name} | {tone_tag}】我是{role}，遇到「{issue}」问题，{tone_q}"
                    )
                    input_ctx = f"{env}；参考工单号：{ticket_code}。"
                    output = (
                        f"您好，关于「{mod_name}」的「{issue}」问题（{tone_tag}），建议按以下步骤处理：\n"
                        f"1. 核心排查与操作：{action}。\n"
                        f"2. 环境适配建议：针对您的「{env}」，请同步确认配置中心与日志目录权限一致。\n"
                        f"3. 服务承诺：{sla}。若按上述步骤仍未解决，请携带工单号 {ticket_code} 联系人工坐席。"
                    )
                    raw_records.append(
                        {
                            "system": SYSTEM_PROMPT,
                            "instruction": instruction,
                            "input": input_ctx,
                            "output": output,
                        }
                    )

    # 额外补充多样化组合确保清洗后仍充足覆盖 target_count
    rng.shuffle(raw_records)
    # 设定较宽松的近似去重阈值(0.95)保留结构化模板下的不同业务槽位组合
    cleaned, _ = clean_alpaca_dataset(raw_records, min_output_len=20, near_dup_threshold=0.95)
    selected = cleaned[:target_count]

    split_idx = int(len(selected) * 0.9)
    return selected[:split_idx], selected[split_idx:]


def export_datasets(output_dir: Path, target_count: int = 500) -> Dict[str, int]:
    """导出 Alpaca 与 ShareGPT 双格式训练集与验证集"""
    output_dir.mkdir(parents=True, exist_ok=True)
    train_set, eval_set = synthesize_customer_service_dataset(target_count=target_count)

    (output_dir / "starmind_cs_train_alpaca.json").write_text(
        json.dumps(train_set, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    (output_dir / "starmind_cs_eval_alpaca.json").write_text(
        json.dumps(eval_set, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    (output_dir / "starmind_cs_train_sharegpt.json").write_text(
        json.dumps(alpaca_to_sharegpt(train_set), ensure_ascii=False, indent=2), encoding="utf-8"
    )
    return {"train": len(train_set), "eval": len(eval_set), "total": len(train_set) + len(eval_set)}


if __name__ == "__main__":
    train_data, eval_data = synthesize_customer_service_dataset(target_count=500)
    print(f"成功合成星辰客服微调语料: 总计={len(train_data) + len(eval_data)} 条 (train={len(train_data)}, eval={len(eval_data)})")
    print("首条样本预览:", json.dumps(train_data[0], ensure_ascii=False))


### 🔍 步骤 2 代码深度解析

这里通过 `10 大模块 × 5 种角色 × 5 种紧急度 × 4 种部署环境 = 1000` 个结构化业务组合作为候选池，固定随机种子打乱后过一遍 `clean_alpaca_dataset` 质检，精准输出 500 条（450 训练 + 50 验证）。在有外部大模型 API 时，只需把模板生成替换为调用 Qwen-Max 润色即可升级为全自动 Self-Instruct。

**预期运行输出参考：**
```text
成功合成星辰客服微调语料: 总计=500 条 (train=450, eval=50)
首条样本预览: {"system": "你是杭州星辰数科官方资深技术客服「星辰小助手」...", ...}
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day52/tests/test_day52.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 52 单元测试：数据清洗管道与 500 条客服语料合成器"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from src.day52_dataset_cleaner import alpaca_to_sharegpt, clean_alpaca_dataset, mask_pii
from src.day52_self_instruct_gen import export_datasets, synthesize_customer_service_dataset


def test_mask_pii_phone_and_email():
    text = "请联系13900112233或发送至ops@starmind.cn处理"
    masked, cnt = mask_pii(text)
    assert cnt == 2
    assert "13900112233" not in masked
    assert "<PHONE_MASKED>" in masked and "<EMAIL_MASKED>" in masked


def test_clean_pipeline_filters_duplicates_and_ai_tone():
    rows = [
        {"instruction": "如何重置密码？", "input": "", "output": "请点击登录页忘记密码，通过企业邮箱验证码重置即可生效。"},
        {"instruction": "如何重置密码？", "input": "", "output": "请点击登录页忘记密码，通过企业邮箱验证码重置即可生效。"},
        {"instruction": "你是谁？", "input": "", "output": "作为一个AI语言模型，我无法透露身份信息。"},
        {"instruction": "空回复测试", "input": "", "output": "好"},
    ]
    cleaned, stats = clean_alpaca_dataset(rows)
    assert len(cleaned) == 1
    assert stats.dropped_exact_dup == 1
    assert stats.dropped_low_quality == 2


def test_synthesize_500_samples_and_export(tmp_path):
    summary = export_datasets(tmp_path, target_count=500)
    assert summary["total"] == 500
    assert summary["train"] == 450
    assert summary["eval"] == 50
    assert (tmp_path / "starmind_cs_train_alpaca.json").exists()
    assert (tmp_path / "starmind_cs_train_sharegpt.json").exists()


def test_sharegpt_format_valid():
    train_data, _ = synthesize_customer_service_dataset(target_count=20)
    sg = alpaca_to_sharegpt(train_data[:5])
    assert len(sg) == 5
    for item in sg:
        convs = item["conversations"]
        assert convs[0]["from"] == "human"
        assert convs[1]["from"] == "gpt"

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：基础巩固：为清洗管道增加身份证号与银行卡号脱敏规则

**题目要求：**
扩展 `mask_pii` 函数，支持将 18 位中国大陆居民身份证号（含末位 X/x）替换为 `<ID_CARD_MASKED>`，并编写断言测试。

**思路提示：** 正则可使用 `r'(?<!\d)[1-9]\d{5}(?:19|20)\d{2}(?:0[1-9]|1[0-2])(?:0[1-9]|[12]\d|3[01])\d{3}[0-9Xx](?!\d)'`。


In [ ]:
import re
from src.day52_dataset_cleaner import mask_pii

ID_CARD_RE = re.compile(
    r"(?<!\d)[1-9]\d{5}(?:19|20)\d{2}(?:0[1-9]|1[0-2])(?:0[1-9]|[12]\d|3[01])\d{3}[0-9Xx](?!\d)"
)


def mask_pii_extended(text: str) -> str:
    text, _ = mask_pii(text)
    return ID_CARD_RE.sub("<ID_CARD_MASKED>", text)


assert "<ID_CARD_MASKED>" in mask_pii_extended("客户证件号33010619950512123X，手机13800138000")
print("身份证与手机号联合脱敏通过！")


**答案解析与评分标准：**

政企客服日志极易泄露身份证号，脱敏必须在数据落盘前执行，防止敏感信息被模型记忆后在推理阶段吐出。


### 作业 2（企业实战）：企业实战：生成 LLaMA-Factory 的 dataset_info.json 注册表

**题目要求：**
编写函数自动生成 LLaMA-Factory 所需的 `dataset_info.json`，同时注册 `starmind_cs_alpaca` 和 `starmind_cs_sharegpt` 两个数据集条目。

**思路提示：** Alpaca 需配置 `columns: {prompt, query, response, system}`；ShareGPT 需配置 `formatting: sharegpt` 与 `tags`。


In [ ]:
import json


def build_dataset_info() -> dict:
    return {
        "starmind_cs_alpaca": {
            "file_name": "starmind_cs_train_alpaca.json",
            "columns": {
                "prompt": "instruction",
                "query": "input",
                "response": "output",
                "system": "system",
            },
        },
        "starmind_cs_sharegpt": {
            "file_name": "starmind_cs_train_sharegpt.json",
            "formatting": "sharegpt",
            "columns": {"messages": "conversations", "system": "system"},
            "tags": {
                "role_tag": "from",
                "content_tag": "value",
                "user_tag": "human",
                "assistant_tag": "gpt",
            },
        },
    }


print(json.dumps(build_dataset_info(), ensure_ascii=False, indent=2))


**答案解析与评分标准：**

注册 `dataset_info.json` 是使用 LLaMA-Factory 时新手最常写错字段名的环节，自动化生成可彻底杜绝 KeyError。


### 作业 3（架构进阶）：架构进阶：构建防灾难性遗忘的通用语料混洗采样器

**题目要求：**
实现函数 `mix_with_general_replay(domain_data, general_data, replay_ratio=0.1)`，按 10% 比例从通用指令集中抽样混入垂直领域训练集，保持最终样本打乱均匀且不修改原列表。

**思路提示：** 计算 `need_general = int(len(domain_data) * replay_ratio / (1 - replay_ratio))`，取 `min(need_general, len(general_data))` 后合并打乱。


In [ ]:
import random
from typing import Dict, List


def mix_with_general_replay(
    domain_data: List[Dict[str, str]],
    general_data: List[Dict[str, str]],
    replay_ratio: float = 0.1,
    seed: int = 42,
) -> List[Dict[str, str]]:
    if not (0.0 <= replay_ratio < 0.5):
        raise ValueError("replay_ratio 建议控制在 [0, 0.5)")
    rng = random.Random(seed)
    need = int(len(domain_data) * replay_ratio / max(1e-6, 1.0 - replay_ratio))
    sampled_general = rng.sample(general_data, min(need, len(general_data)))
    mixed = list(domain_data) + sampled_general
    rng.shuffle(mixed)
    return mixed


res = mix_with_general_replay([{"id": i} for i in range(90)], [{"id": f"g{i}"} for i in range(50)], 0.1)
print(f"混洗后总样本数: {len(res)}")


**答案解析与评分标准：**

混入 5%~10% 通用高质量指令数据（如 Belle/COIG 精选）是业界最成熟的防灾难性遗忘手段，既保留领域专业度又不损害通用推理能力。


---

## 🎯 老陈收尾复盘

漂亮！500 条星辰客服高质量微调语料已经落盘，重复率 0%，格式合规率 100%，还顺带切好了 90% 训练集和 10% 验证集。记住：以后在公司做微调，花 70% 时间在数据清洗和 Bad Case 构造上，花 30% 时间调参，效果绝对吊打只会调学习率的同行。
